<a href="https://colab.research.google.com/github/RomanLuja/ML/blob/main/project_kickoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ппроєкт: від Kaggle-датасету до першої моделі

## 1. Проєкт: що і як

**Завдання:** побудувати прогнозну модель на реальному датасеті з Kaggle: очистити дані, створити
ознаки, порівняти моделі з коректною валідацією, пояснити модель і написати висновок для бізнесу.

**Датасети на вибір** (усі публічні, завантажуються через `kagglehub` без реєстрації):

| № | Датасет | Задача | Чим цікавий |
|---|---|---|---|
| 1 | [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) | класифікація: чи піде клієнт | числа, записані як текст; бізнес-розрахунок утримання |
| 2 | [Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand) | класифікація: чи скасують бронювання | **витоки цілі**, дублікати, валідація за часом, 119 тис. рядків |
| 3 | [New York City Airbnb](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data) | регресія: ціна за ніч | викиди, пропуски, багато категорій, логарифм цілі |

Складність зростає від 1 до 3

---
## Практика: старт власного проєкту

1. Відкрийте `project_template.ipynb`, збережіть копію у свій Google Drive
   (`Файл → Зберегти копію на Диску`).
2. Оберіть датасет і змініть `DATASET` і `FILE` у комірці завантаження.
3. Заповніть **розділ 1** (постановка). Покажіть викладачу до кінця заняття.
4. Почніть **розділ 2** (аудит): типи колонок, пропуски, дублікати, ціль, витоки.

**Підказки до датасетів (без спойлерів):**
- **Telco:** перевірте тип кожної «числової» колонки — чи справді вона число?
- **Hotel:** для кожної колонки запитайте: «Чи відомо це в момент бронювання?» Подумайте, як
  розділити дані, щоб модель прогнозувала майбутні бронювання.
- **Airbnb:** подивіться на мінімальні й максимальні ціни. Чи всі оголошення реальні?

| Слово | Простими словами |
|---|---|
| Таблиця, DataFrame | дані в pandas: рядки (об'єкти, наприклад працівники) і колонки (їхні властивості) |
| Ціль (target, y) | що хочемо передбачити: звільниться людина чи ні |
| Ознаки (features, X) | усе, на основі чого модель робить прогноз: вік, зарплата, переробки… |
| Train / test | дані, на яких модель вчиться / дані, які ховаємо до кінця, щоб чесно перевірити модель |
| Модель | програма, яка за прикладами знайшла правило «ознаки → ціль» |
| fit / predict | навчити модель на прикладах / отримати прогноз для нових даних |
| ROC-AUC | оцінка якості від 0.5 (вгадування) до 1.0 (ідеально): наскільки добре модель ставить ризикових людей вище |
| random_state=42 | «зерно» випадковості: з ним кожен запуск дає ті самі результати |

## 2. Демонстрація: проєкт у мініатюрі

Пройдемо всі розділи шаблону швидко на датасеті [IBM HR Analytics Employee Attrition](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)
(його немає серед варіантів проєкту). Мета — побачити **порядок дій**, а не ідеальний результат.

### 0. Завантаження з Kaggle

| Columns | Values |
| ------- | ------ |
| Age | Вік працівника |


In [1]:
import os  # робота з файлами й теками: подивитися, що лежить у теці, склеїти шлях до файлу
import subprocess  # запуск команд, ніби ви ввели їх у терміналі
import sys  # відомості про поточний Python

import numpy as np  # швидкі обчислення з масивами чисел
import pandas as pd  # таблиці: читання CSV, фільтри, групування
import matplotlib.pyplot as plt  # графіки

try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
print(os.listdir(path))
ibm = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-HR-Employee-Attrition.csv"))
print(ibm.shape)
ibm.head(3)

Using Colab cache for faster access to the 'ibm-hr-analytics-attrition-dataset' dataset.
['WA_Fn-UseC_-HR-Employee-Attrition.csv']
(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


### 1. Постановка

- **Замовник:** HR; рішення — кому запропонувати програму утримання.
- **Ціль:** `Attrition` (Yes/No) → класифікація; один рядок — працівник.
- **Метрика:** ROC-AUC (наскільки добре модель ставить ризикових людей угору списку) + recall (яку частку всіх звільнень ми знаходимо) у топ-15 % списку; бізнес — вартість заміни працівника.

### 2. Аудит

In [2]:
ibm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [3]:
ibm['Attrition'].value_counts()

,count
Attrition,
No,1233
Yes,237


In [4]:
print("Дублікати:", ibm.duplicated().sum())
print("Пропуски:", ibm.isna().sum().sum())
print("Частка звільнень:", (ibm['Attrition'] == 'Yes').mean().round(3))

Дублікати: 0
Пропуски: 0
Частка звільнень: 0.161


Частка звільнень: 0.161 (16.1 %). Орієнтир «завжди ні» — 83.9 % правильних відповідей

In [5]:
print('Ідентифікатор: EmployeeNumber', ibm['EmployeeNumber'].is_unique)

Ідентифікатор: EmployeeNumber True


In [6]:
constant = [c for c in ibm.columns if ibm[c].nunique() == 1]
print('Константи:', constant)

Константи: ['EmployeeCount', 'Over18', 'StandardHours']


In [7]:
df = ibm.drop(columns=constant + ['EmployeeNumber'])
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)

In [8]:
df.head(3)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0


### 3. Ознаки

In [9]:
df['IncomePerLevel'] = df['MonthlyIncome'] / df['JobLevel']
df['YearsWithoutPromotionShare'] = df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1)
df['CompaniesPerYear'] = df['NumCompaniesWorked'] / (df['TotalWorkingYears'] + 1)

### 4. Моделювання і валідація

- `train_test_split` — розділити дані на train і test.
- `cross_val_score`, `StratifiedKFold` — крос-валідація: оцінка моделі на кількох частинах train.
- `ColumnTransformer`, `make_column_selector` — різна обробка для числових і текстових колонок.
- `Pipeline` — ланцюжок «обробка → модель» як один об'єкт.
- `OneHotEncoder` — текстову категорію перетворює на колонки 0/1; `StandardScaler` — масштабує числа.
- `DummyClassifier` — модель-орієнтир; `LogisticRegression` — логістична регресія; `HistGradientBoostingClassifier` — градієнтний бустинг.
- `roc_auc_score` — метрика ROC-AUC.

In [10]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

In [11]:
X = df.drop(columns='Attrition')
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train — {len(y_train)} людей, test — {len(y_test)} людини (з них звільнилися {(y_test.sum())})")

Train — 1176 людей, test — 294 людини (з них звільнилися 47)


In [12]:
def prep(scale):
    return ColumnTransformer([
        ('num', StandardScaler() if scale else 'passthrough', make_column_selector(dtype_include='number')),
        ('cat', OneHotEncoder(handle_unknown='ignore'), make_column_selector(dtype_exclude='number'))
    ])

# ColumnTransformer([
#         ('num', StandardScaler(), ['n1', 'n2', 'n3']),
#         ('cat', OneHotEncoder(handle_unknown='ignore'), ['s1', 's2'])
#     ])


- `def prep(scale):` — оголошуємо функцію з параметром `scale` (True або False).
- Рядок `"num"`: для всіх числових колонок (`dtype_include="number"`) — `StandardScaler()`, якщо `scale=True`, або `"passthrough"` (без змін), якщо `False`. Логістичній регресії масштаб потрібен, деревам і бустингу — ні.
- `A if умова else B` — короткий запис «якщо умова, то A, інакше B».
- Рядок `"cat"`: для всіх нечислових колонок — `OneHotEncoder`: категорія «Sales» стає колонкою `Department_Sales` зі значеннями 1/0.
- `handle_unknown="ignore"` — якщо в нових даних з'явиться невідома категорія, модель не зламається.

In [13]:
candidates = {
    "Орієнтир": Pipeline([
                    ('prep', prep(False)),
                    ('model', DummyClassifier(strategy='prior'))
                ]),
    "Логістична регресія": Pipeline([
                    ('prep', prep(True)),
                    ('model', LogisticRegression(max_iter=2000))
                ]),
    "Бустинг": Pipeline([
                    ('prep', prep(False)),
                    ('model', HistGradientBoostingClassifier(learning_rate=0.05, random_state=42))
                ])
}

- Словник `{назва: значення}` дозволяє потім перебрати всі моделі одним циклом.
- `Pipeline([("prep", ...), ("model", ...)])` — два кроки з іменами `prep` (обробка) і `model` (модель clf - класифікатор). Під час навчання обробка теж вчиться **лише на train** — без «підглядання» в test.
- **Орієнтир** `DummyClassifier(strategy="prior")` дає всім однакову ймовірність (частку звільнень), тобто «вгадує». Його ROC-AUC завжди 0.5 — нижня межа.
- **Логістична регресія** — проста лінійна модель; `max_iter=2000` — дозволяємо їй більше кроків навчання, щоб вона встигла «зійтися».
- **Бустинг** — багато дерев рішень, кожне виправляє помилки попередніх; `learning_rate=0.05` — обережні кроки.

In [14]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)

journal = []
for name, model in candidates.items():
    sc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
    journal.append({"модель": name, "AUC CV": sc.mean(), "std": sc.std()})

pd.DataFrame(journal).round(3)

,модель,AUC CV,std
0,Орієнтир,0.500,0.000
1,Логістична регресія,0.840,0.028
2,Бустинг,0.804,0.028


Навчаємо логістичну  модель - кращий показник

In [15]:
final = candidates["Логістична регресія"].fit(X_train, y_train)
proba = final.predict_proba(X_test)[:, 1]  # y_predict
print(f"Test AUC: {roc_auc_score(y_test, proba):.3f}")

Test AUC: 0.816


- `predict_proba` повертає дві колонки: ймовірність класу 0 і класу 1.
- `[:, 1]` — «усі рядки, друга колонка», тобто ймовірність звільнення.



In [16]:
# Бізнес-метрика: топ-15 % списку
# Беремо 15 % людей з найвищим ризиком і рахуємо, скільки з них справді звільнилися.

top = np.argsort(-proba)[: int(0.15 * len(proba))]
print(f"У топ-15% списку - {y_test.iloc[top].sum()} з {y_test.sum()} тих, хто звільнився ({y_test.iloc[top].sum() / y_test.sum():.0%})")

# У топ-15% списку - 80 з 100 тих, хто звільнився (80%)

У топ-15% списку - 23 з 47 тих, хто звільнився (49%)


- `np.argsort(-proba)` — номери людей, упорядковані від найвищого ризику до найнижчого (мінус розвертає сортування).
- `int(0.15 * len(proba))` — скільки людей становлять 15 % (44 з 294); `[: ...]` — перші стільки номерів.
- `y_te.iloc[top].sum()` — скільки з обраних справді звільнилися; `y_te.sum()` — скільки звільнилося всього.
- `:.0%` — формат у відсотках без десяткових знаків.

### 5. Інтерпретація

Оцінюємо, на скільки кожна ознака важлива для моделі.

- Ідея: перемішуємо значення **однієї** колонки в test (ламаємо її зв'язок з ціллю) і дивимося, наскільки впав ROC-AUC. Сильно впав — ознака важлива.
- `n_repeats=10` — перемішуємо 10 разів і усереднюємо, щоб результат не залежав від одного випадкового перемішування.
- `n_jobs=-1` — використовувати всі ядра процесора (швидше).

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(final, X_test, y_test, scoring="roc_auc", n_repeats=10,
                            random_state=42, n_jobs=-1)

pd.Series(pi.importances_mean, index=X_test.columns).sort_values().tail(8).plot.barh(title="Permutation importance")
plt.show()

pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False).head(8)


### Результат

> Висновок (для замовника)